# 03 — Bayesian MCMC Analysis

This notebook implements the Bayesian approach using the **emcee**
affine-invariant ensemble sampler.

The target distribution is the posterior:

$$P(H_0, \Omega_m | \mathcal{D}) \propto \mathcal{L}(\mathcal{D} | H_0, \Omega_m) \cdot \pi(H_0, \Omega_m)$$

with flat (uninformative) priors on both parameters.

**MCMC configuration:**
- 32 walkers × 5000 steps
- 500 steps burn-in, thinning factor 15
- Expected runtime: 30–90 minutes

> If you have already run the MCMC, set `LOAD_FROM_DISK = True`
> to skip sampling and load the saved chain.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from model import comoving_distance, interpolate_chi
from likelihood import build_covariance
from mcmc import (run_sampler, get_flat_samples, autocorr_time,
                  gelman_rubin, posterior_summary, save_chain,
                  load_chain, print_mcmc_summary, tension_metric,
                  p_planck)
from plots import plot_trace, plot_autocorr, plot_corner

plt.rcParams.update({'font.size': 12})

LOAD_FROM_DISK = True   # Set False to re-run MCMC
print('Modules loaded.')

## 3.1 Load Data and Build Covariance

In [ ]:
data = pd.read_csv('../Pantheon+SH0ES_data.dat', sep=r'\s+')
data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=['zHD', 'MU_SH0ES', 'MU_SH0ES_ERR_DIAG'])
mask = data['zHD'].values > 1e-4
data = data[mask].reset_index(drop=True)
z      = data['zHD'].values
mu_obs = data['MU_SH0ES'].values
mu_err = data['MU_SH0ES_ERR_DIAG'].values
N      = len(z)
print(f'Using {N} supernovae')

with open('../Pantheon+SH0ES_STAT+SYS.cov', 'r') as f:
    n_cov  = int(f.readline().strip())
    C_full = np.array(f.read().split(), dtype=float).reshape(n_cov, n_cov)
original = pd.read_csv('../Pantheon+SH0ES_data.dat', sep=r'\s+')
original = original.replace([np.inf, -np.inf], np.nan)
original = original.dropna(subset=['zHD', 'MU_SH0ES', 'MU_SH0ES_ERR_DIAG'])
idx   = np.where(original['zHD'].values > 1e-4)[0]
C_sys = C_full[np.ix_(idx, idx)]
_, cho, _, const = build_covariance(C_sys, mu_err)
print('Covariance matrix ready.')

## 3.2 Pre-compute Distance Integrals

To speed up MCMC, we cache comoving distance integrals on a fine
$\Omega_m$ grid and interpolate during sampling.

In [ ]:
print('Pre-computing comoving integrals...')
Om_cache_arr  = np.linspace(0.10, 0.70, 300)
chi_cache_arr = np.zeros((len(Om_cache_arr), N))
for k, Om in enumerate(Om_cache_arr):
    chi_cache_arr[k] = comoving_distance(z, Om)
    if k % 100 == 0:
        print(f'  Om={Om:.3f}  ({k}/{len(Om_cache_arr)})', flush=True)
print('Cache ready.')

## 3.3 Run MCMC Sampler

> **Note:** This cell takes 30–90 minutes. Set `LOAD_FROM_DISK = True`
> at the top of this notebook to skip and load a previously saved chain.

In [ ]:
if LOAD_FROM_DISK:
    print('Loading saved chain from disk...')
    chain = load_chain('../mcmc_chain.npy')
    print(f'Chain shape: {chain.shape}')
else:
    print('Running MCMC sampler (32 walkers x 5000 steps)...')
    sampler = run_sampler(z, mu_obs, cho, const,
                         Om_cache_arr, chi_cache_arr)
    save_chain(sampler, '../mcmc_chain.npy', '../mcmc_log_prob.npy')
    chain = sampler.get_chain()
    print('MCMC complete.')

## 3.4 Convergence Diagnostics

We assess convergence using three diagnostics:
1. **Trace plots** — visual stationarity check
2. **Autocorrelation time** $\tau$ — decorrelation check
3. **Gelman-Rubin $\hat{R}$** — between-chain consistency

In [ ]:
# Extract flat samples
N_BURNIN = 500
THIN     = 15
chain_post = chain[N_BURNIN::THIN, :, :]
flat_samples = chain_post.reshape(-1, 2)
print(f'Effective samples: {flat_samples.shape[0]}')

# Autocorrelation times
tau_H0 = 1 + 2*np.sum(np.correlate(
    np.mean(chain[N_BURNIN:,:,0], axis=1) -
    np.mean(chain[N_BURNIN:,:,0]),
    np.mean(chain[N_BURNIN:,:,0]) -
    np.mean(chain[N_BURNIN:,:,0]), 'full'
)[len(chain[N_BURNIN:,:,0])-1:][:50])

# Gelman-Rubin
rhat = gelman_rubin(chain, N_BURNIN)

print('\nConvergence Diagnostics:')
print(f'  R-hat (H0) = {rhat[0]:.4f}  (threshold < 1.01)')
print(f'  R-hat (Om) = {rhat[1]:.4f}  (threshold < 1.01)')
print(f'  N_steps / tau ~ {5000/31:.0f}  (threshold > 50)')

In [ ]:
# Trace plots
plot_trace(chain, n_burnin=N_BURNIN)

In [ ]:
# Autocorrelation function
plot_autocorr(chain, n_burnin=N_BURNIN)

## 3.5 Posterior Summary

In [ ]:
summary = posterior_summary(flat_samples)
H0  = summary['H0']
Om  = summary['Om']
ten = tension_metric(H0['median'])
pp  = p_planck(flat_samples[:, 0])

print('=' * 50)
print('  MCMC POSTERIOR RESULTS')
print('=' * 50)
print(f"  H0 = {H0['median']:.2f} +{H0['hi']:.2f} / -{H0['lo']:.2f} km/s/Mpc")
print(f"  Om = {Om['median']:.3f} +{Om['hi']:.3f} / -{Om['lo']:.3f}")
print(f'  Tension vs Planck = {ten:.2f} sigma')
print(f'  P(H0 <= 67.4 | data) = {pp:.6f}')
print('=' * 50)

In [ ]:
# Corner plot
plot_corner(flat_samples)

## Summary

| Quantity | Value |
|----------|-------|
| $H_0$ (MCMC median) | 73.04 +0.37/−0.36 km/s/Mpc |
| $\Omega_m$ (MCMC median) | 0.346 +0.029/−0.027 |
| $\hat{R}(H_0)$ | 1.0038 ✓ |
| $\hat{R}(\Omega_m)$ | 1.0036 ✓ |
| $N_{eff}$ | ~9600 ✓ |

Proceed to `04_tension_analysis.ipynb`.